In [1]:
import sys
import os
import torch
import pandas as pd
from IPython.display import display, HTML

BASE_DIR = "/content/multilingual_bias"

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    if os.path.isdir(BASE_DIR):
        import shutil
        print("Removing existing repo to update")
        shutil.rmtree(BASE_DIR)
  
    from git import Repo
    
    os.chdir("/")

    repo_url = "https://github.com/jonesmonez/multilingual_bias"
    Repo.clone_from(repo_url, BASE_DIR, branch="python312", single_branch=True)
    
    os.chdir(BASE_DIR)
    print(f"Repository cloned to {BASE_DIR}")

In [3]:
from experiments.modules.crows_runner import CrowSPairsRunnerWrapper

runner = CrowSPairsRunnerWrapper()

to_test_types = ["gender", "race", "religion"]
# to_test_lang = ["ar_DZ", "ca_ES", "de_DE", "en_US", "es_AR", "fr_FR", "it_IT", "mt_MT", "zh_CN"]
to_test_lang = ["ar_DZ", "ca_ES", "de_DE", "en_US", "es_AR", "fr_FR", "mt_MT", "zh_CN"]
debias_lang = ["en_US", "de_DE", "fr_FR"]

In [ ]:
results = {}

for lang in to_test_lang:
    result_btype = {}
    
    for btype in to_test_types:
        
        result = runner.run_plain(
            path_to_crows=f"data/crows_improved/crows_{lang}.csv",
            lang_eval=lang,
            bias_type=btype,
        )
        
        result_btype[btype] = result[0]
    
    results[lang] = result_btype
    
print(results)
display(pd.DataFrame.from_dict(results, orient="index"))

In [ ]:
results = {}

for dlang in debias_lang:
    result_lang = {}
    
    for lang in to_test_lang:
        result_btype = {}
        
        for btype in to_test_types:
            
            result = runner.run_debias(
                path_to_crows=f"data/crows_improved/crows_{lang}.csv",
                lang_debias=dlang,
                lang_eval=lang,
                bias_type=btype,
                bias_direction=f"data/subspace/subspace_m-BertModel_c-bert-base-multilingual-uncased_t-{btype}_debias-{dlang}.pt",
            )
            
            result_btype[btype] = result[0]
        
        result_lang[lang] = result_btype
    
    results[dlang] = result_lang
    
for lang, data in results.items():
    print(data)
    display(pd.DataFrame.from_dict(data, orient="index"))